In [5]:
!pip install torch torchvision
!pip install opencv-python

In [6]:
import torch
import torch.nn as nn
import torchvision
from torchvision import datasets, models, transforms
from torch.utils.data import DataLoader

In [7]:
from google.colab import files
uploaded = files.upload()

Saving dataset.zip to dataset.zip


In [8]:
!unzip dataset.zip


Archive:  dataset.zip
   creating: dataset/normal/
  inflating: dataset/normal/000001.jpg  
  inflating: dataset/normal/000002.jpg  
  inflating: dataset/normal/000003.jpg  
  inflating: dataset/normal/000004.jpg  
  inflating: dataset/normal/000005.jpg  
  inflating: dataset/normal/000006.jpg  
  inflating: dataset/normal/000007.jpg  
  inflating: dataset/normal/000008.jpg  
  inflating: dataset/normal/000009.jpg  
  inflating: dataset/normal/000010.jpg  
  inflating: dataset/normal/000011.jpg  
  inflating: dataset/normal/000012.jpg  
  inflating: dataset/normal/000013.jpg  
  inflating: dataset/normal/000014.jpg  
  inflating: dataset/normal/000015.jpg  
  inflating: dataset/normal/000016.jpg  
  inflating: dataset/normal/000017.jpg  
  inflating: dataset/normal/000018.jpg  
  inflating: dataset/normal/000019.jpg  
  inflating: dataset/normal/000020.jpg  
  inflating: dataset/normal/000021.jpg  
  inflating: dataset/normal/000022.jpg  
  inflating: dataset/normal/000023.jpg  
  infl

In [9]:
transform = transforms.Compose([
    transforms.Resize((224,224)),
    transforms.ToTensor()
])

In [10]:
dataset = datasets.ImageFolder("dataset", transform=transform)

train_size = int(0.8 * len(dataset))
test_size = len(dataset) - train_size

train_data, test_data = torch.utils.data.random_split(dataset, [train_size, test_size])

train_loader = DataLoader(train_data, batch_size=16, shuffle=True)
test_loader = DataLoader(test_data, batch_size=16)

In [11]:
model = models.resnet50(pretrained=True)

model.fc = nn.Linear(model.fc.in_features, 2)


/usr/local/lib/python3.12/dist-packages/torchvision/models/_utils.py:208: UserWarning: The parameter 'pretrained' is deprecated since 0.13 and may be removed in the future, please use 'weights' instead.
  warnings.warn(
/usr/local/lib/python3.12/dist-packages/torchvision/models/_utils.py:223: UserWarning: Arguments other than a weight enum or `None` for 'weights' are deprecated since 0.13 and may be removed in the future. The current behavior is equivalent to passing `weights=ResNet50_Weights.IMAGENET1K_V1`. You can also use `weights=ResNet50_Weights.DEFAULT` to get the most up-to-date weights.
  warnings.warn(msg)


Downloading: "https://download.pytorch.org/models/resnet50-0676ba61.pth" to /root/.cache/torch/hub/checkpoints/resnet50-0676ba61.pth


100%|██████████| 97.8M/97.8M [00:00<00:00, 145MB/s]


In [12]:
criterion = nn.CrossEntropyLoss()
optimizer = torch.optim.Adam(model.parameters(), lr=0.001)

In [13]:
for epoch in range(5):

    for images, labels in train_loader:

        outputs = model(images)
        loss = criterion(outputs, labels)

        optimizer.zero_grad()
        loss.backward()
        optimizer.step()

    print("Epoch:", epoch+1, "Loss:", loss.item())

Epoch: 1 Loss: 0.20263142883777618
Epoch: 2 Loss: 0.04599564149975777
Epoch: 3 Loss: 0.11699157953262329
Epoch: 4 Loss: 0.013580298982560635
Epoch: 5 Loss: 0.1198778972029686


In [14]:
correct = 0
total = 0

with torch.no_grad():

    for images, labels in test_loader:

        outputs = model(images)
        _, predicted = torch.max(outputs, 1)

        total += labels.size(0)
        correct += (predicted == labels).sum().item()

print("Accuracy:", 100 * correct / total)


Accuracy: 92.85714285714286


In [15]:
from google.colab import files
uploaded = files.upload()

Saving abc.jpg to abc (1).jpg


In [16]:
from PIL import Image

img = Image.open("abc.jpg")

img = transform(img).unsqueeze(0)

output = model(img)

prediction = torch.argmax(output)

if prediction == 1:
    print("⚠ Possible Facial Paralysis")
else:
    print("✅ Normal Face")

⚠ Possible Facial Paralysis
